# 26 - Reading contract 3: what stacking real lights settled

**Purpose.** Explain what notebook `25` measured by registering and stacking the sky-pair night,
and what it changes, for someone deciding what to do next. `25` is written for someone *checking*
it; this one is slow on purpose.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `stack_constants.json`, `stack_ladder.csv`, `stack_pairs.csv`, and the files they are
compared against. If anything here disagrees with `results/`, `results/` is right and this notebook
is the bug.

**It assumes** `00_statistics.ipynb` for the statistics, `18_sky_pair_read.ipynb` for the night
itself, and `22_pi_arithmetic_read.ipynb` for why every loss in a stack is the combination and
never the arithmetic.

**The headline, recomputed in section 1 rather than typed here:** the definition of done is met as
MISSION writes it, and the model still misses in one direction every time in green.

In [ ]:
import json
import pathlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import model as M

RESULTS = pathlib.Path.cwd().parent / "results"
c = M.Constants.load(RESULTS / "stack_constants.json")
ladder = pd.read_csv(RESULTS / "stack_ladder.csv")
pairs = pd.read_csv(RESULTS / "stack_pairs.csv")
bias = json.loads((RESULTS / "dark_constants.json").read_text())["eta_comb"]["value"]

## 1. The verdict

MISSION's test: the model must rank at least three pairs correctly, to within 10%, with one pair
straddling the HCG threshold at gain 200 - and each pair must be one the model predicts *apart*,
by more than the SNR measurement's own repeatability. Green is the plane the predictions were
made in.

In [ ]:
green = pairs[pairs.plane == "green"]
print(green[["pair", "why", "predicted_pct", "measured_pct", "repeatability_pct", "verdict"]]
      .round(2).to_string(index=False))
passed = green[green.verdict == "pass"]
ties = (green.verdict == "tie").sum()
straddle = passed.why.str.contains("straddles").any()
print(f"\n{len(passed)} of {len(green)} pass, {ties} ties, a pass straddles HCG: {straddle}")
print("definition of done:", "MET" if len(passed) >= 3 and straddle else "NOT MET")

### What that does and does not say

**No pair was a tie.** Every predicted separation clears the repeatability - the closest, gain 50
to 200 at 120 s, by a factor of 1.75 - so each of the four is a real test rather than one every
model passes. That was the risk MISSION named, and it did not happen here.

**The pass bar is loose, and a reader should see how loose.** "Within 10%" is on the *ratio*: a
predicted +27% against a measured +18% is 1.176 / 1.273 = 7.6% apart, and passes. The repeatability
is 2-3%. So a pass means "right direction, right rough size", not "right to the measurement's
precision". The next cell shows the misses at that precision.

In [ ]:
p = pairs.copy()
p["miss_points"] = p.measured_pct - p.predicted_pct
p["miss_in_sigmas"] = p.miss_points / p.repeatability_pct
print(p[["pair", "plane", "predicted_pct", "measured_pct", "miss_points", "miss_in_sigmas", "verdict"]]
      .round(2).to_string(index=False))
g = p[p.plane == "green"]
print(f"\ngreen: every miss has the same sign: {bool((np.sign(g.miss_points) == -1).all())}; "
      f"mean miss {g.miss_points.mean():+.1f} points")
b = p[p.plane == "B"]
print(f"blue:  mean miss {b.miss_points.mean():+.1f} points, range "
      f"{b.miss_points.min():+.1f} to {b.miss_points.max():+.1f}")

## 2. The model over-promises in green, every time

In green the model predicts a larger gain than the stack delivers in all four pairs, by 1.8 to 4.2
times the repeatability. The one failure - gain 50 to gain 200 at 120 s - is not a different kind of
miss; it is the same shortfall landing on the pair whose predicted gain was smallest, so it
crosses zero and the winner flips.

**A miss that always has the same sign is a missing term, not noise.** Every pair here moves
towards less read noise - a longer sub, or the high-gain mode - and the model's gain from that move
comes entirely from `R^2/t` shrinking against the sky. If some noise does *not* shrink with `t`,
the model credits the move with more than it can buy. That is the first of MISSION's four
assumptions, *no noise term that fails to scale with t*, and it is the one still untested.

**This notebook cannot settle it, and says so.** Two other candidates would push the same way:
`F_obj` is dropped from the predicted noise (the signal box holds nebula, so it is not zero), and
the dark current used is the -10 C bound on a night shot at -20 C. Why blue tracks the model better
than green is a clue for whoever takes this up - not something to explain away here.

## 3. `eta_comb` on real lights, beside the bias ladder

In [ ]:
reg = c.entry("eta_comb_registered")
none = c.entry("eta_comb_registered_no_rejection")
print("N    registered, winsorized   registered, no rejection   bias ladder (session 03)")
for n in reg["value"]:
    print(f"{int(n):3d}   {reg['value'][n]:.4f} +- {reg['uncertainty'][n]:.4f}"
          f"        {none['value'][n]:.4f} +- {none['uncertainty'][n]:.4f}"
          f"          {bias.get(n, '-')}")

**Dithered lights do not stall the way the bias stack did.** The bias ladder fell to 0.88 by N=16,
because a bias carries a fixed pattern that sits on the same pixel in every frame and cannot average
away. Dithering moves the sky across the sensor between frames, so after registration that fixed
pattern lands on a different sky pixel each time and averages like noise. The bias ladder was
always an upper bound on the *loss*; on real, dithered lights most of that loss is simply not there.

**Rejection costs a few percent at these N, and buys nothing visible back.** The winsorized column
sits below the no-rejection one at every rung: whatever it removed from this night's signal box
was worth less than the real pixels it discarded with it. That is a statement about this night's
box, not a recommendation to stop rejecting - one satellite trail through it would change the sign.

**The no-rejection column goes above 1, and that is a flaw in the ideal, not a gift.** No stack
beats `sqrt(N)`. The ideal here is the noise of a pair of *registered single frames*, and a single
pair carries a little that is not noise: a star imaged under different seeing, or registered 0.5
px off, does not cancel exactly in the difference. A stack of N averages those over N frames, so
the stack loses that excess and the single pair keeps it - which inflates the ideal and, with it,
every efficiency on this page. The no-rejection arm puts a size on it: up to about 5%. **So read
the published `eta_comb_registered` as high by up to that much**, and the true combination cost as
the winsorized column minus a few percent. That is still well above the bias ladder at N=16.

**The top of the ladder is N=18.** A night of 15 s subs is hundreds of frames; nothing here says
whether the efficiency holds there, and the model must still refuse to extrapolate.

## 4. Resampling: measured, and deliberately not divided out

In [ ]:
rf = c.entry("resampling_factor")
print("registered / raw single-frame noise:", {k: round(v, 4) for k, v in rf["value"].items()})
diag = ladder[ladder.arm == "winsorized"].groupby("n")[["eta_comb_4", "eta_vs_raw_4"]].mean()
print("\nwinsorized, 4x4, mean over cells and planes:")
print(diag.round(4).to_string())

Registration resamples every frame onto the reference grid, and resampling mixes each output pixel
from its neighbours. Per pixel that reads about 10% quieter than the raw frame; at 4x4 about 5%.
The noise has not gone anywhere - a kernel that sums to one moves noise between neighbours - it has
been spread over a scale the per-pixel spread cannot see.

**The obvious correction was tried, and it is the column on the right.** Dividing the efficiency by
the resampling factor measures the stack against a *raw* sub, which is what the model's `SNR_sub`
describes - and it comes out above 1, a stack quieter than `sqrt(N)` allows. The raw frames, lined
up by a whole-pixel shift, carry about 10% more 4x4 noise than white noise would, and this night
cannot say whether that is sky that did not cancel or correlated noise the sensor really has. Either
way it inflates the raw ideal. So the published `eta_comb` uses the registered frame as its ideal,
and resampling is treated as a wash at the scale of extended signal.

**If that extra 4x4 noise turns out to be the sensor's**, it is the same thing section 2 is looking
for: noise that does not average down like white noise. Two sections point at the same suspect.

## 5. How repeatable the SNR estimator is

In [ ]:
rep = c.entry("snr_repeatability")
print("relative error of one cell's full-stack SNR, %, mean over planes:", rep["value"])

Two estimates were taken and the larger used, a rule fixed before any number was seen: the
protocol's half-split, which is a single draw, and the scatter of four quarter-stacks, which has
three degrees of freedom. Neither was allowed to be the one that happened to come out smaller.

About 1-3% per cell, and a pair's bar is its two cells combined. That is small against every
predicted separation here, which is why no pair was a tie - and large against nothing the model
currently gets wrong, which is why section 2's misses are real.

## 6. What is settled, and what is next

**Settled.**
- The split approach registers every plane of every frame of this night, blue included, without
  debayering anything. The engine is no longer the blocker.
- `eta_comb` on registered, dithered lights is published at 0.96-1.02 up to N=18, high by up to
  about 5% (section 3): close to ideal either way, and far better than the bias ladder the model
  was carrying.
- MISSION's definition of done is met as written: three of four green pairs pass, one straddling
  HCG, and none of them a tie.

**Not settled, and the next measurement is the one that would settle it.**
1. **The one-way miss in green.** A noise term that does not shrink with `t` is the leading
   suspect, and it is MISSION's last untested assumption. It is a model question before it is a
   bench question: put `F_obj` and the -20 C dark current into the prediction first, and see how
   much of the miss is left.
2. **`eta_comb` at N of hundreds.** Needs a night of many short subs, or the archive.
3. **Star clipping per star** (rule 7) - the star-colour half of the Pareto point - now possible,
   since the frames can be registered.